# A Strands agent on Bedrock with Notion's MCP server, through the Highflame AI gateway

An [AWS Strands](https://strandsagents.com) agent on an Amazon Bedrock model, with the tools of
[Notion's hosted MCP server](https://developers.notion.com/docs/mcp). Every model call goes through
the **Highflame AI gateway**. The agent reaches Notion directly, over OAuth.

| Traffic | Path |
| --- | --- |
| Model calls | agent → **AI gateway** → Bedrock |
| Notion sign-in (OAuth) | agent and your browser ↔ Notion's authorization server |
| MCP calls (`tools/list`, `tools/call`) | agent → `mcp.notion.com` |

**What the gateway sees.** Each prompt, each tool call the model asks for (Notion's tools
included), each tool result the agent sends back to the model, and each reply. It checks them and
records them against your API key. Nothing is registered, neither the agent nor the MCP server.

**What it does not see.** The MCP requests between the agent and Notion. The gateway does not
accept or forward an OAuth token it did not issue: MCP forbids that token passthrough, because a
token minted for one audience must not be replayed to another. To route the MCP traffic through
Highflame as well, register the server in the MCP gateway and do the OAuth there, as
[`strands_bedrock_mcp_via_mcp_gateway.ipynb`](strands_bedrock_mcp_via_mcp_gateway.ipynb) does.

## Setup

### 1. Add Bedrock as a provider on the AI gateway

**Studio → AI Gateway → LLM Providers → AWS Bedrock.** Set **Bedrock Endpoint** to the runtime
endpoint of the region you will call, for example `https://bedrock-runtime.us-east-1.amazonaws.com`.

The gateway holds no AWS credential and signs nothing. It forwards a **Bedrock API key** to Bedrock
as a bearer token, and the setup cell makes one for you: a short-term key minted from the AWS
credentials you already use (`AWS_PROFILE`, SSO, environment variables). The AWS identity needs
`bedrock:CallWithBearerToken` and permission to invoke the model. Prefer a long-term key from the
Bedrock console? Put it in `BEDROCK_API_KEY` and nothing is minted.

A model is named `bedrock/<model id>`, such as
`bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0`. The gateway serves Bedrock on its
OpenAI-compatible endpoint, so the agent uses Strands' `OpenAIModel` pointed at the gateway, not
`BedrockModel`, which calls AWS directly.

### 2. Notion

A Notion account is all you need. Notion's MCP server supports dynamic client registration, so the
notebook registers itself as an OAuth client when you sign in, and there is no app to create.

**Run this notebook on your own machine.** The sign-in opens your browser and Notion redirects back
to `http://localhost:8765/callback`, where the notebook listens. A remote kernel cannot receive that
redirect.

### 3. The `.env` file

Copy `.env.example` to `.env` beside this notebook.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** Studio → AI Gateway → Settings → API Keys. Authenticates to the gateway. |
| `HIGHFLAME_GATEWAY_BASE_URL` | **Required.** `https://gateway.highflame.ai/llm/v1`, or your deployment's LLM base URL with `/v1`. |
| `AWS_REGION` | Optional. The region of the Bedrock endpoint from step 1. Unset, your AWS profile's region is used. |
| `AWS_PROFILE` | Optional. The AWS profile to mint the Bedrock API key from. |
| `BEDROCK_MODEL_ID` | Optional. Defaults to `us.anthropic.claude-sonnet-4-5-20250929-v1:0`. |
| `BEDROCK_API_KEY` | Optional. A Bedrock API key to use instead of minting one. |
| `MCP_SERVER_URL` | Optional. Defaults to `https://mcp.notion.com/mcp`. |
| `OAUTH_CALLBACK_PORT` | Optional. Defaults to `8765`. |

Note: You can use this notebook for other providers too.

In [ ]:
%pip install -q -r requirements.txt

from importlib.metadata import version

for package in ("strands-agents", "mcp", "highflame"):
    print(package, version(package))

In [ ]:
import asyncio
import collections
import os

from dotenv import load_dotenv
from openai import OpenAI, OpenAIError
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient

load_dotenv() 

HIGHFLAME_API_KEY = os.environ["HIGHFLAME_API_KEY"]
GATEWAY_BASE_URL = os.environ["HIGHFLAME_GATEWAY_BASE_URL"]
AWS_REGION = os.environ.get("AWS_REGION")
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID") or "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
MODEL = f"bedrock/{BEDROCK_MODEL_ID}"

QUESTION = "Search my Notion workspace and list the titles of up to three pages you can find, with one line about each."

# The gateway forwards this to Bedrock as `Authorization: Bearer`. A minted key lasts up to 12
# hours, and never longer than the AWS session it came from.
BEDROCK_API_KEY = os.environ.get("BEDROCK_API_KEY")
if not BEDROCK_API_KEY:
    from aws_bedrock_token_generator import provide_token

    BEDROCK_API_KEY = provide_token(region=AWS_REGION)


def gateway_model() -> OpenAIModel:
    """Bedrock, reached through the Highflame AI gateway.

    `api_key` is sent as `Authorization: Bearer` and forwarded to Bedrock. The Highflame key travels
    in its own header, so the two credentials never compete for one."""
    return OpenAIModel(
        client_args={
            "base_url": GATEWAY_BASE_URL,
            "api_key": BEDROCK_API_KEY,
            "default_headers": {"X-Highflame-APIKey": HIGHFLAME_API_KEY},
        },
        model_id=MODEL,
        params={"max_tokens": 1024, "temperature": 0},
    )

REFUSAL_MARKERS = ("Highflame Security:", "was blocked by security policy")


def show(result) -> None:
    """Print the agent's answer, whether Highflame refused a step, and which tools ran."""
    text = str(result).strip()
    print(("Refused by Highflame: " if any(m in text for m in REFUSAL_MARKERS) else "") + text)
    print("\ntools the agent ran:", ", ".join(result.metrics.tool_metrics) or "none")


try:
    probe = OpenAI(
        base_url=GATEWAY_BASE_URL, api_key=BEDROCK_API_KEY, default_headers={"X-Highflame-APIKey": HIGHFLAME_API_KEY}
    ).chat.completions.create(
        model=MODEL, max_tokens=5, messages=[{"role": "user", "content": "Reply with the single word: ok"}]
    )
except OpenAIError as exc:
    raise RuntimeError(
        f"The gateway did not answer a Bedrock call: {exc}\n"
        "Check HIGHFLAME_GATEWAY_BASE_URL and HIGHFLAME_API_KEY, that AWS Bedrock is added under "
        "Studio -> AI Gateway -> LLM Providers for AWS_REGION, and that your AWS identity may call the model."
    ) from exc
if probe.id.startswith("chatcmpl-blocked-"):
    raise RuntimeError(f"The gateway refused a plain prompt: {probe.choices[0].message.content!r}")

print("gateway:", GATEWAY_BASE_URL)
print("model  :", MODEL, "in", AWS_REGION)
print("probe  :", probe.choices[0].message.content)

gateway: https://gateway.highflame.ai/llm/v1
model  : bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0 in us-east-1
probe  : ok


## 1. The model, through the gateway

A plain turn first, with no tools. The model call leaves this process for the gateway, which
checks the prompt, records the call against your API key and forwards it to Bedrock. The reply
comes back the same way.

In [2]:
agent = Agent(model=gateway_model(), callback_handler=None)
show(await agent.invoke_async("In one sentence, what is the Model Context Protocol?"))

The Model Context Protocol (MCP) is an open standard that enables AI models to securely connect to external data sources and tools through a client-server architecture.

tools the agent ran: none


## 2. Sign in to Notion, directly

This is the only part of the run in which the agent talks to Notion for anything but tools, and
none of it passes through Highflame:

1. The first MCP request comes back `401`, naming Notion's protected-resource metadata.
2. The MCP client reads that metadata and Notion's authorization-server metadata.
3. It registers itself as an OAuth client (dynamic client registration).
4. Your browser opens Notion's consent page, with PKCE. Approve access to the pages you choose.
5. Notion redirects to `localhost` with a code, and the client exchanges it for a token.

The token stays in this process's memory. It is not written to disk and not sent to Highflame.

Every request the MCP client sends is also written to `MCP_TRAFFIC`, so the next cells can show
where each one went. An httpx auth hook is the one place Strands exposes each outgoing request, so
the recorder wraps the MCP SDK's OAuth provider rather than changing it.

The agent gets Notion's **read-only** tools only. Nothing here writes to your workspace.

In [ ]:
import queue
import threading
import webbrowser
import http.server
import urllib

from mcp.client.auth import OAuthClientProvider
from mcp.shared.auth import AuthorizationCodeResult, OAuthClientMetadata

MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL") or "https://mcp.notion.com/mcp"
CALLBACK_PORT = int(os.environ.get("OAUTH_CALLBACK_PORT") or 8765)
REDIRECT_URI = f"http://localhost:{CALLBACK_PORT}/callback"
READ_ONLY_TOOLS = [
    "notion-search", 
    "notion-fetch", 
    "notion-get-users", 
    "notion-get-teams", 
    "notion-get-comments"
]
MCP_TRAFFIC: list[tuple[str, str, str]] = []  # (method, host, path) of every request the MCP client sent


class InMemoryTokenStorage:
    """Holds the OAuth client registration and the token for this process only."""

    tokens = client_info = None

    async def get_tokens(self):
        return self.tokens

    async def set_tokens(self, tokens) -> None:
        self.tokens = tokens

    async def get_client_info(self):
        return self.client_info

    async def set_client_info(self, client_info) -> None:
        self.client_info = client_info


class RecordedOAuth(OAuthClientProvider):
    """The MCP SDK's OAuth provider, unchanged, plus a record of each request it lets out."""

    async def async_auth_flow(self, request):
        flow = super().async_auth_flow(request)
        try:
            outgoing = await flow.__anext__()
            while True:
                MCP_TRAFFIC.append((outgoing.method, outgoing.url.host, outgoing.url.path))
                response = yield outgoing
                outgoing = await flow.asend(response)
        except StopAsyncIteration:
            return


# Notion redirects the browser here with the authorization code.
codes: "queue.Queue[dict[str, str]]" = queue.Queue()


class CallbackHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        url = urllib.parse.urlparse(self.path)
        if url.path != "/callback":
            self.send_response(404)
            self.end_headers()
            return
        codes.put(dict(urllib.parse.parse_qsl(url.query)))
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.end_headers()
        self.wfile.write(b"<p>Signed in to Notion. You can close this tab and go back to the notebook.</p>")

    def log_message(self, *args):  # keep the notebook output quiet
        pass


previous_server = globals().get("callback_server")
if previous_server is not None:  # the cell was re-run: free the port first
    previous_server.shutdown()
    previous_server.server_close()
callback_server = http.server.HTTPServer(("127.0.0.1", CALLBACK_PORT), CallbackHandler)
threading.Thread(target=callback_server.serve_forever, daemon=True).start()


async def open_browser(authorization_url: str) -> None:
    print("Opening Notion's sign-in page. If no browser opens, visit:\n", authorization_url)
    webbrowser.open(authorization_url)


async def wait_for_code() -> AuthorizationCodeResult:
    params = await asyncio.to_thread(codes.get, True, 300)  # five minutes to approve
    if "error" in params:
        raise RuntimeError(f"Notion did not grant access: {params.get('error_description') or params['error']}")
    return AuthorizationCodeResult(code=params["code"], state=params.get("state"), iss=params.get("iss"))


oauth = RecordedOAuth(
    server_url=MCP_SERVER_URL,
    client_metadata=OAuthClientMetadata(
        client_name="Highflame cookbook: Strands + Notion",
        redirect_uris=[REDIRECT_URI],
        grant_types=["authorization_code", "refresh_token"],
        response_types=["code"],
        token_endpoint_auth_method="none",  # a public client: PKCE, no client secret
    ),
    storage=InMemoryTokenStorage(),
    redirect_handler=open_browser,
    callback_handler=wait_for_code,
)

# startup_timeout covers the sign-in: the connection opens only once you approve in the browser.
notion = MCPClient(
    url=MCP_SERVER_URL, auth_provider=oauth, startup_timeout=300, tool_filters={"allowed": READ_ONLY_TOOLS}
)
notion.start()
notion_tools = notion.list_tools_sync()
print("\nconnected to", MCP_SERVER_URL)
print("read-only tools the agent gets:", ", ".join(t.tool_name for t in notion_tools) or "none")

In [4]:
MCP_PATH = urllib.parse.urlparse(MCP_SERVER_URL).path


def summarize_traffic() -> None:
    """Where each request the MCP client sent went, and whether it was OAuth or MCP."""
    counts = collections.Counter(
        (host, "MCP" if path == MCP_PATH else f"OAuth {path}") for _, host, path in MCP_TRAFFIC
    )
    for (host, kind), n in counts.items():
        print(f"  {host:24} {kind:52} {n}")


print("Requests the MCP client sent so far:")
summarize_traffic()
print("\nThe consent page itself ran in your browser, so it does not appear here.")

Requests the MCP client sent so far:
  mcp.notion.com           MCP                                                  7
  mcp.notion.com           OAuth /.well-known/oauth-protected-resource/mcp      1
  mcp.notion.com           OAuth /.well-known/oauth-authorization-server        1
  mcp.notion.com           OAuth /register                                      1
  mcp.notion.com           OAuth /token                                         1

The consent page itself ran in your browser, so it does not appear here.


## 3. The agent: model through the gateway, Notion tools direct

One agent, both halves. When the model asks for a Notion tool, the reply carrying that tool call
comes through the gateway, which checks the call before the agent sees it. The agent then runs the
tool against Notion directly, and sends the result back to the model through the gateway, which
checks the result too.

In [ ]:
notion_agent = Agent(
    model=gateway_model(),
    tools=notion_tools,
    system_prompt="You help the user find things in their Notion workspace. Use the Notion tools to answer.",
    callback_handler=None,  # print the answer once, below, instead of streaming it
)
show(await notion_agent.invoke_async(QUESTION))

## Clean up

This stops the MCP connection and the local callback server.

The notebook registered one OAuth client with Notion, and you granted it access. To revoke that
access, open Notion's **Settings → Connections** and disconnect **Highflame cookbook: Strands +
Notion**.

In [6]:
notion.stop(None, None, None)
callback_server.shutdown()
callback_server.server_close()
print("stopped")

stopped
